# 10 · Every result against the study's

Each model's development and held-out average precision, set against the study's own
(`reference/expected_results.csv` and `expected_study.csv`). A full run on the study's software
versions reproduces them exactly; a model the study did not finish (most of CatBoost) is listed
as such. The study's reading of all these numbers, with intervals and paired comparisons, is in
`research_record/`.

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

In [ ]:
ref = repro.reference("expected_results.csv")
rows = []
for r in ref.itertuples():
    p = lib.cell_path(P.results, r.set, r.rung, r.model)
    if not p.exists():
        rows.append({"set": r.set, "rung": r.rung, "model": r.model, "status": "not run"}); continue
    m = lib.load_cell(P.results, r.set, r.rung, r.model)["meta"]
    rows.append({"set": r.set, "rung": r.rung, "model": r.model, "status": "done",
                 "held_AP": m["held_metrics"]["AP"], "study_held_AP": r.held_AP,
                 "difference": m["held_metrics"]["AP"] - r.held_AP})
res = pd.DataFrame(rows)
print(res.status.value_counts().to_string())
ok = res[res.status == "done"]
repro.check(f"{len(ok)} library models match the study's held-out AP (1e-6)",
            len(ok) == len(res) and ok.difference.abs().max() <= 1e-6, f"largest difference {ok.difference.abs().max():.1e}")

In [ ]:
repro.compare_cells(P.project / "study", ["Logistic regression", "Random Forest", "XGBoost"], sets=("s21",),
                    ref_name="expected_study.csv")

In [ ]:
# the leaderboard at each rung: the best tuned models on any set
best = ok.merge(ref[["set", "rung", "model", "n_predictors"]], on=["set", "rung", "model"])
for rung in lib.RUNGS:
    print(rung)
    print(best[best.rung == rung].sort_values("held_AP", ascending=False).head(10)[["set", "model", "n_predictors", "held_AP"]]
          .round(3).to_string(index=False), "\n")
repro.report()